In [1]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy import stats as spstats
import numpy as np

class FC_DimRed_Gio(TransformerMixin, BaseEstimator):
    """
    Feature reduction class based on functional connectivity (FC) matrices.
    Selects discriminative nodes using the Kruskal-Wallis test and ranks nodes by their degree.

    Parameters:
    - threshold: p-value threshold for Kruskal-Wallis test
    - nb_nodes: number of nodes to select
    """
    def __init__(self, p_threshold = 0.05, eta_threshold = 0.10, nb_nodes = 78):
        self.p_threshold = p_threshold
        self.nb_nodes = nb_nodes
        self.eta_threshold = eta_threshold

    def fit(self, X, y, metric='p-value'):
        """
        Fit the model by identifying discriminative nodes based on the Kruskal-Wallis test.

        Parameters:
        - X: 3D array (samples, channels, channels) representing FC matrices
        - y: 1D array of class labels
        - metric: 'p-value' or 'eta-squared'
        """
        # Split data by class
        unique_classes = np.unique(y)
        FC_classes = [X[np.where(y == cls)] for cls in unique_classes]

        # Kruskal-Wallis test
        H, pvals = spstats.kruskal(*FC_classes, axis=0)

        if metric == 'p-value':
            # Binarize edges based on p-value threshold
            thresh_mask = (pvals < self.p_threshold).astype(int)
        if metric == 'eta-squared':
            # Compute eta-squared
            eta_squared = (H - 4 + 1) / (X.shape[0] - 4) 
            # Binarize edges based on eta-squared threshold
            thresh_mask = (eta_squared > self.eta_threshold).astype(int)

        # Compute node degree and select top nodes
        node_strength = np.sum(thresh_mask, axis=0)
        self.node_strength_ = node_strength 
        idx = np.argsort(-node_strength)
        self.node_select_ = idx[:self.nb_nodes]

        return self

    def transform(self, X):
        """
        Transform the input data by selecting the top-ranked nodes.

        Parameters:
        - X: 3D array (samples, channels, channels) representing FC matrices

        Returns:
        - Transformed 3D array with reduced dimensions
        """
        return X[:, self.node_select_, :][:, :, self.node_select_]

## 

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from pyriemann.classification import TSClassifier
import numpy as np
import os
import sys

if os.getcwd()[-6:] == "v3_Gio":
    os.chdir("../")
project_root = os.getcwd()  # should be .../TURING
src_path = os.path.join(project_root, "v2", "src")

sys.path.append(src_path)

from models.ensemble import EnsembleClassifier

sys.path.append(os.path.abspath("../v2/data/features/CovarianceMatrices/OAS/MCI"))


from utils.utils import load_cov_mats, load_atms, load_corr_mats


import warnings
import os
import json
warnings.filterwarnings('ignore')

# Load the data
# TODO: selection in cross validation loop !

# Models definition
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_atm, y_atm = load_atms(zscore=1.6)
X_atm = X_atm[:, :78, :78]  
dim_red_eta_20_atm = FC_DimRed_Gio(eta_threshold=0.1, nb_nodes=20)
X_atm = dim_red_eta_20_atm.fit_transform(X_atm, y_atm, metric='eta-squared')
print(f"Selected nodes ATMs: {dim_red_eta_20_atm.node_select_}")



X_cov_mat, y_cov_mat = load_cov_mats()
X_cov_mat = X_cov_mat[:, :78, :78]
dim_red_eta_50_cov = FC_DimRed_Gio(eta_threshold=0.1, nb_nodes=50)
X_cov_mat = dim_red_eta_50_cov.fit_transform(X_cov_mat, y_cov_mat, metric='eta-squared')
print(f"Selected nodes Covariance Matrices: {dim_red_eta_50_cov.node_select_}")


X_corr_mat, y_corr_mat = load_corr_mats()
X_corr_mat = X_corr_mat[:, :78, :78]  
dim_red_eta_20_corr = FC_DimRed_Gio(eta_threshold=0.1, nb_nodes=20)
X_corr_mat = dim_red_eta_20_corr.fit_transform(X_corr_mat, y_corr_mat, metric='eta-squared')
print(f"Selected nodes Correlation Matrices: {dim_red_eta_20_corr.node_select_}")


Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)
Selected nodes ATMs: [56 36 57 38 63 64 49 15 14 54 25 52 11  6 65  1 59 42 17 51]
Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)
Selected nodes Covariance Matrices: [77 11 73 51 33 26 62 50 27 14 36 67 72 18 44 37 34 15  5 71  8 12 24 31
  9 39 38 23 45 63 76  7 65 70 13 40 25 17 20 75 56  1 43 59 32 21 22 52
 66 47]
Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)
Selected nodes Correlation Matrices: [77 67 20 38 56 17 50 51 12 62 11  6 75 73 54 45 36 18  7 65]


Previous selection (original code Mario)
Selected nodes Covariance Matrices: [ 1  5  7  8  9 11 12 13 14 15 17 18 20 21 22 23 24 25 26 27 31 32 33 34
 36 37 38 39 40 43 44 45 50 51 52 56 59 60 62 63 65 66 67 70 71 72 73 75
 76 77]
Selected nodes ATMs: [ 1  6 11 14 15 17 25 36 38 42 47 49 52 54 56 57 59 63 64 65]

In [3]:
dim_red_eta_20_corr.node_strength_[dim_red_eta_20_corr.node_select_]

array([26, 15, 13, 13, 13, 11,  9,  9,  8,  8,  7,  7,  7,  7,  7,  6,  6,
        6,  6,  6])

In [3]:
dim_red_eta_20_atm.node_strength_[dim_red_eta_20_atm.node_select_]

array([14, 14, 13, 12, 12, 12, 11, 11, 11, 10, 10, 10,  9,  9,  9,  8,  8,
        8,  8,  7])

In [4]:
dim_red_eta_50_cov.node_strength_[dim_red_eta_50_cov.node_select_]

array([51, 39, 38, 36, 34, 34, 33, 33, 33, 33, 32, 32, 32, 31, 31, 31, 31,
       31, 31, 31, 31, 31, 30, 29, 28, 28, 28, 28, 27, 27, 27, 26, 26, 26,
       26, 26, 25, 25, 25, 25, 25, 25, 24, 24, 24, 23, 22, 22, 22, 21])

In [6]:
ROI_AAL_list = np.array([ 'Rectus_L','Olfactory_L','Frontal_Sup_Orb_L','Frontal_Med_Orb_L','Frontal_Mid_Orb_L',
                'Frontal_Inf_Orb_L','Frontal_Sup_L','Frontal_Mid_L','Frontal_Inf_Oper_L','Frontal_Inf_Tri_L',
                'Frontal_Sup_Medial_L','Supp_Motor_Area_L','Paracentral_Lobule_L','Precentral_L','Rolandic_Oper_L',
                'Postcentral_L','Parietal_Sup_L','Parietal_Inf_L','SupraMarginal_L','Angular_L','Precuneus_L',
                'Occipital_Sup_L','Occipital_Mid_L','Occipital_Inf_L','Calcarine_L','Cuneus_L','Lingual_L',
                'Fusiform_L','Heschl_L','Temporal_Sup_L','Temporal_Mid_L','Temporal_Inf_L','Temporal_Pole_Sup_L',
                'Temporal_Pole_Mid_L','ParaHippocampal_L','Cingulum_Ant_L','Cingulum_Mid_L','Cingulum_Post_L',
                'Insula_L','Rectus_R','Olfactory_R','Frontal_Sup_Orb_R','Frontal_Med_Orb_R','Frontal_Mid_Orb_R',
                'Frontal_Inf_Orb_R','Frontal_Sup_R','Frontal_Mid_R','Frontal_Inf_Oper_R','Frontal_Inf_Tri_R',
                'Frontal_Sup_Medial_R','Supp_Motor_Area_R','Paracentral_Lobule_R','Precentral_R','Rolandic_Oper_R',
                'Postcentral_R','Parietal_Sup_R','Parietal_Inf_R', 'SupraMarginal_R','Angular_R','Precuneus_R',
                'Occipital_Sup_R','Occipital_Mid_R','Occipital_Inf_R','Calcarine_R','Cuneus_R','Lingual_R',
                'Fusiform_R','Heschl_R','Temporal_Sup_R','Temporal_Mid_R','Temporal_Inf_R','Temporal_Pole_Sup_R',
                'Temporal_Pole_Mid_R','ParaHippocampal_R','Cingulum_Ant_R','Cingulum_Mid_R','Cingulum_Post_R',
                'Insula_R','Hippocampus_L','Hippocampus_R','Amygdala_L','Amygdala_R','Caudate_L','Caudate_R',
                'Putamen_L','Putamen_R','Pallidum_L','Pallidum_R','Thalamus_L','Thalamus_R','Cerebelum_Crus1_L',
                'Cerebelum_Crus1_R','Cerebelum_Crus2_L','Cerebelum_Crus2_R','Cerebelum_3_L','Cerebelum_3_R',
                'Cerebelum_4_5_L','Cerebelum_4_5_R','Cerebelum_6_L','Cerebelum_6_R','Cerebelum_7b_L','Cerebelum_7b_R',
                'Cerebelum_8_L','Cerebelum_8_R','Cerebelum_9_L','Cerebelum_9_R','Cerebelum_10_L','Cerebelum_10_R',
                'Vermis_1_2','Vermis_3','Vermis_4_5','Vermis_6','Vermis_7','Vermis_8','Vermis_9','Vermis_10'])

In [7]:
ROI_AAL_list[dim_red_eta_20_corr.node_select_]

array(['Insula_R', 'Heschl_R', 'Precuneus_L', 'Insula_L',
       'Parietal_Inf_R', 'Parietal_Inf_L', 'Supp_Motor_Area_R',
       'Paracentral_Lobule_R', 'Paracentral_Lobule_L', 'Occipital_Inf_R',
       'Supp_Motor_Area_L', 'Frontal_Sup_L', 'Cingulum_Mid_R',
       'ParaHippocampal_R', 'Postcentral_R', 'Frontal_Sup_R',
       'Cingulum_Mid_L', 'SupraMarginal_L', 'Frontal_Mid_L', 'Lingual_R'],
      dtype='<U20')

In [8]:
ROI_AAL_list[dim_red_eta_20_atm.node_select_]

array(['Parietal_Inf_R', 'Cingulum_Mid_L', 'SupraMarginal_R', 'Insula_L',
       'Calcarine_R', 'Cuneus_R', 'Frontal_Sup_Medial_R', 'Postcentral_L',
       'Rolandic_Oper_L', 'Postcentral_R', 'Cuneus_L', 'Precentral_R',
       'Supp_Motor_Area_L', 'Frontal_Sup_L', 'Lingual_R', 'Olfactory_L',
       'Precuneus_R', 'Frontal_Med_Orb_R', 'Parietal_Inf_L',
       'Paracentral_Lobule_R'], dtype='<U20')

In [9]:
ROI_AAL_list[dim_red_eta_50_cov.node_select_]

array(['Insula_R', 'Supp_Motor_Area_L', 'ParaHippocampal_R',
       'Paracentral_Lobule_R', 'Temporal_Pole_Mid_L', 'Lingual_L',
       'Occipital_Inf_R', 'Supp_Motor_Area_R', 'Fusiform_L',
       'Rolandic_Oper_L', 'Cingulum_Mid_L', 'Heschl_R',
       'Temporal_Pole_Mid_R', 'SupraMarginal_L', 'Frontal_Inf_Orb_R',
       'Cingulum_Post_L', 'ParaHippocampal_L', 'Postcentral_L',
       'Frontal_Inf_Orb_L', 'Temporal_Pole_Sup_R', 'Frontal_Inf_Oper_L',
       'Paracentral_Lobule_L', 'Calcarine_L', 'Temporal_Inf_L',
       'Frontal_Inf_Tri_L', 'Rectus_R', 'Insula_L', 'Occipital_Inf_L',
       'Frontal_Sup_R', 'Calcarine_R', 'Cingulum_Post_R', 'Frontal_Mid_L',
       'Lingual_R', 'Temporal_Inf_R', 'Precentral_L', 'Olfactory_R',
       'Cuneus_L', 'Parietal_Inf_L', 'Precuneus_L', 'Cingulum_Mid_R',
       'Parietal_Inf_R', 'Olfactory_L', 'Frontal_Mid_Orb_R',
       'Precuneus_R', 'Temporal_Pole_Sup_L', 'Occipital_Sup_L',
       'Occipital_Mid_L', 'Precentral_R', 'Fusiform_R',
       'Frontal_In